In [0]:
%sql
show catalogs;

In [0]:
%sql
show schemas in workspace;

In [0]:
%sql
create Schema if not exists workspace.bronze;
create schema if not exists workspace.silver;
create schema if not exists workspace.gold;

In [0]:
%sql
create table if not exists workspace.bronze.po_raw (
    source_row_id int,
    po_id string,
    po_line_no int,
    supplier_id string,
    quantity int,
    approved_quantity int,
    unit_price Decimal(10,2)
);

In [0]:
%sql
create table if not exists workspace.bronze.supplier_raw(
    supplier_id string,
    supplier_name string
);

In [0]:
%sql
INSERT INTO workspace.bronze.po_raw VALUES
    (1, 'PO1001', 1, 'SUP101', 10, 10, 250.00),
    (2, 'PO1002', 1, 'SUP102',  5,  5, 100.00),
    (3, 'PO1003', 1, 'SUP999',  4,  6, 150.00),
    (4, 'PO1004', 1, 'SUP101', -2,  5, 200.00),
    (5, 'PO1005', 1, 'SUP102',  8,  8,  50.00),
    (6, 'PO1005', 1, 'SUP102',  8,  8,  50.00),
    (7, 'PO1006', 1, NULL,      3,  3,  75.00),
    (8, 'PO1007', 1, 'SUP101',  4,  2, 120.00),
    (9, 'PO1008', 1, 'SUP102',  6,  6,  -5.00),
    (10, NULL,    1, 'SUP101',  2,  2, 100.00);

In [0]:
%sql
INSERT INTO workspace.bronze.supplier_raw VALUES
    ('SUP101', 'Supplier A'),
    ('SUP102', 'Supplier B'),
    ('SUP103', 'Supplier C');

In [0]:
%sql
select * from workspace.bronze.po_raw
order by source_row_id;

In [0]:
%sql
select * from workspace.bronze.supplier_raw;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.business_rules (
    rule_id          STRING,
    rule_name        STRING,
    rule_type        STRING,
    severity         STRING,
    rule_description STRING
);

In [0]:
%sql
INSERT INTO workspace.silver.business_rules VALUES
('DQ01', 'PO ID mandatory',
 'DATA_QUALITY', 'CRITICAL',
 'Purchase order ID must not be null'),

('DQ02', 'Supplier ID mandatory',
 'DATA_QUALITY', 'HIGH',
 'Supplier ID must not be null'),

('DQ03', 'Unique PO line',
 'DATA_QUALITY', 'CRITICAL',
 'PO ID and line number must form a unique key'),

('DQ04', 'Positive quantity',
 'DATA_QUALITY', 'HIGH',
 'Ordered quantity must be greater than zero'),

('DQ05', 'Non-negative unit price',
 'DATA_QUALITY', 'HIGH',
 'Unit price must be zero or greater'),

('BR01', 'Approved quantity check',
 'BUSINESS', 'HIGH',
 'Ordered quantity must not exceed approved quantity'),

('BR02', 'Valid supplier reference',
 'BUSINESS', 'HIGH',
 'Supplier must exist in the supplier master');

In [0]:
%sql
SELECT *
FROM workspace.silver.business_rules
ORDER BY rule_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.po_staging AS
SELECT
    p.*,

    COUNT(*) OVER (
        PARTITION BY po_id, po_line_no
    ) AS po_line_count,

    CASE
        WHEN s.supplier_id IS NOT NULL THEN TRUE
        ELSE FALSE
    END AS supplier_exists

FROM workspace.bronze.po_raw p

LEFT JOIN workspace.bronze.supplier_raw s
    ON p.supplier_id = s.supplier_id;

In [0]:
%sql
select * from workspace.silver.po_staging;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.po_rule_results (
    source_row_id     INT,
    po_id             STRING,
    supplier_id       STRING,
    rule_id           STRING,
    failed            BOOLEAN,
    rule_name         STRING,
    severity          STRING,
    rule_description  STRING
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.po_rule_results AS

WITH checks AS (

    -- DQ01: PO ID must not be null
    SELECT
        source_row_id, po_id, supplier_id,
        'DQ01' AS rule_id,
        po_id IS NULL AS failed
    FROM workspace.silver.po_staging

    UNION ALL

    -- DQ02: Supplier ID must not be null
    SELECT
        source_row_id, po_id, supplier_id,
        'DQ02',
        supplier_id IS NULL
    FROM workspace.silver.po_staging

    UNION ALL

    -- DQ03: PO line key must be unique
    SELECT
        source_row_id, po_id, supplier_id,
        'DQ03',
        po_line_count > 1
    FROM workspace.silver.po_staging

    UNION ALL

    -- DQ04: Quantity must be positive
    SELECT
        source_row_id, po_id, supplier_id,
        'DQ04',
        quantity <= 0
    FROM workspace.silver.po_staging

    UNION ALL

    -- DQ05: Unit price must be non-negative
    SELECT
        source_row_id, po_id, supplier_id,
        'DQ05',
        unit_price < 0
    FROM workspace.silver.po_staging

    UNION ALL

    -- BR01: Quantity must not exceed approved quantity
    SELECT
        source_row_id, po_id, supplier_id,
        'BR01',
        quantity > approved_quantity
    FROM workspace.silver.po_staging

    UNION ALL

    -- BR02: Supplier must exist in the supplier master
    SELECT
        source_row_id, po_id, supplier_id,
        'BR02',
        NOT supplier_exists
    FROM workspace.silver.po_staging
)

SELECT
    c.source_row_id,
    c.po_id,
    c.supplier_id,
    c.rule_id,
    c.failed,
    r.rule_name,
    r.severity,
    r.rule_description
FROM checks c
JOIN workspace.silver.business_rules r
    ON c.rule_id = r.rule_id;

In [0]:
%sql
SELECT COUNT(*) AS total_rule_evaluations
FROM workspace.silver.po_rule_results;

In [0]:
%sql
SELECT
    source_row_id,
    po_id,
    supplier_id,
    rule_id,
    rule_name,
    severity
FROM workspace.silver.po_rule_results
WHERE failed = TRUE
ORDER BY source_row_id, rule_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.po_exceptions AS
SELECT *
FROM workspace.silver.po_rule_results
WHERE failed = TRUE;

In [0]:
%sql
SELECT *
FROM workspace.silver.po_exceptions
ORDER BY source_row_id, rule_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.po_validated AS
SELECT p.*
FROM workspace.bronze.po_raw p
LEFT ANTI JOIN (
    SELECT DISTINCT source_row_id
    FROM workspace.silver.po_exceptions
) e
ON p.source_row_id = e.source_row_id;

In [0]:
%sql
SELECT COUNT(*) AS validated_records
FROM workspace.silver.po_validated;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dq_summary AS
SELECT
    rule_id,
    rule_name,
    severity,
    COUNT(*) AS records_checked,
    SUM(CASE WHEN failed THEN 1 ELSE 0 END) AS failures,
    SUM(CASE WHEN NOT failed THEN 1 ELSE 0 END) AS passes,
    ROUND(
        100.0 * SUM(CASE WHEN failed THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS failure_pct
FROM workspace.silver.po_rule_results
GROUP BY
    rule_id,
    rule_name,
    severity;

In [0]:
%sql
SELECT *
FROM workspace.gold.dq_summary
ORDER BY failures DESC, rule_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dq_overall_summary AS
SELECT
    COUNT(*) AS total_checks,
    SUM(CASE WHEN failed THEN 1 ELSE 0 END) AS failed_checks,
    SUM(CASE WHEN NOT failed THEN 1 ELSE 0 END) AS passed_checks,
    ROUND(
        100.0 * SUM(CASE WHEN NOT failed THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS overall_pass_pct
FROM workspace.silver.po_rule_results;

In [0]:
%sql
SELECT *
FROM workspace.gold.dq_overall_summary;

### 3.3 Create a run summary

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.dq_run_summary (
    run_id STRING,
    run_timestamp TIMESTAMP,
    total_checks BIGINT,
    failed_checks BIGINT,
    passed_checks BIGINT,
    overall_pass_pct DECIMAL(5,2)
);

In [0]:
%sql
INSERT INTO workspace.gold.dq_run_summary
SELECT
    'POV_RUN_001' AS run_id,
    CURRENT_TIMESTAMP() AS run_timestamp,
    total_checks,
    failed_checks,
    passed_checks,
    CAST(overall_pass_pct AS DECIMAL(5,2))
FROM workspace.gold.dq_overall_summary;

### Verify the complete gold layer

In [0]:
%sql
SELECT *
FROM workspace.gold.dq_run_summary;

In [0]:
%sql
SELECT * FROM workspace.gold.dq_overall_summary;

In [0]:
%sql
SELECT * FROM workspace.gold.dq_run_summary;